# 15. Log-Mel CNN — MP3 Compression Robustness

13번에서 학습한 **in-domain Log-Mel CNN**을 재학습하지 않고,
동일한 Test set을 다음 세 조건에서 평가한다.

- `original`: 현재 데이터셋에 저장된 기준 파일
- `mp3_128`: MP3 128 kbps로 재인코딩
- `mp3_64`: MP3 64 kbps로 재인코딩

핵심 원칙:

```text
13번 best CNN checkpoint 그대로 사용
13번 Validation EER threshold 그대로 사용
Test Original / 128k / 64k만 입력 조건 변경
재학습 없음
```

12번에서 이미 만든 MP3 재인코딩 파일을 재사용한다.

## 1. 경로 / 라이브러리 / Device 설정

In [ ]:
from pathlib import Path
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    roc_curve,
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
)

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

ORIGINAL_LOGMEL_PATH = (
    PROJECT_ROOT / "data/processed/logmel/logmel_10s_float16.npy"
)
ORIGINAL_DONE_PATH = (
    PROJECT_ROOT / "data/processed/logmel/logmel_10s_done.npy"
)

COMPRESSED_ROOT = PROJECT_ROOT / "data/processed/mp3_robustness"

CNN_CHECKPOINT = PROJECT_ROOT / "checkpoints/cnn/logmel_cnn_best.pt"
CNN_THRESHOLD_PATH = PROJECT_ROOT / "results/cnn/cnn_thresholds.json"

SVM_MP3_METRICS = (
    PROJECT_ROOT / "results/mp3_robustness/mp3_robustness_metrics.csv"
)

CNN_MP3_CACHE_DIR = PROJECT_ROOT / "data/processed/logmel/mp3_robustness"
CNN_MP3_RESULT_DIR = PROJECT_ROOT / "results/cnn_mp3_robustness"

CNN_MP3_CACHE_DIR.mkdir(parents=True, exist_ok=True)
CNN_MP3_RESULT_DIR.mkdir(parents=True, exist_ok=True)

SR = 24_000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)

N_FFT = 1024
HOP_LENGTH = 240
N_MELS = 128
FMAX = 12_000
N_FRAMES = 1 + TARGET_SAMPLES // HOP_LENGTH

BATCH_SIZE = 8
RANDOM_STATE = 42

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)
print("Segment:", SEGMENT_PATH)
print("MP3 root:", COMPRESSED_ROOT)

## 2. 재현성 설정

In [ ]:
def seed_everything(seed=RANDOM_STATE):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

## 3. 입력 파일 / Test 구조 QC

In [ ]:
required = [
    SEGMENT_PATH,
    ORIGINAL_LOGMEL_PATH,
    ORIGINAL_DONE_PATH,
    CNN_CHECKPOINT,
    CNN_THRESHOLD_PATH,
]

for p in required:
    print(p.name, "->", p.exists())

segments = pd.read_csv(SEGMENT_PATH).reset_index(drop=True)
test_segments = segments[segments["split"] == "test"].copy()
test_segments["global_index"] = test_segments.index
test_segments = test_segments.reset_index(drop=True)

test_tracks = (
    test_segments
    .drop_duplicates("track_sample_id")
    .reset_index(drop=True)
)

original_cache = np.load(ORIGINAL_LOGMEL_PATH, mmap_mode="r")
original_done = np.load(ORIGINAL_DONE_PATH)

print("\nAll segment rows :", len(segments))
print("Test segments    :", len(test_segments))
print("Test tracks      :", len(test_tracks))
print("Original cache   :", original_cache.shape)
print("Original done    :", int(original_done.sum()), "/", len(original_done))

assert len(segments) == 10077
assert len(test_segments) == 1572
assert len(test_tracks) == 539
assert original_cache.shape == (10077, 128, 1001)
assert bool(original_done.all())

print("Test structure QC PASS: True")

## 4. 12번에서 만든 MP3 파일 존재 여부 확인

In [ ]:
def compressed_path(condition, track_sample_id):
    return (
        COMPRESSED_ROOT
        / condition
        / f"{track_sample_id}.mp3"
    )

rows = []

for condition in ["mp3_128", "mp3_64"]:
    exists_count = 0
    missing = []

    for track_id in test_tracks["track_sample_id"]:
        p = compressed_path(condition, track_id)
        if p.exists() and p.stat().st_size > 0:
            exists_count += 1
        else:
            missing.append(str(track_id))

    rows.append({
        "condition": condition,
        "expected_tracks": len(test_tracks),
        "existing_tracks": exists_count,
        "missing_tracks": len(missing),
    })

mp3_file_qc = pd.DataFrame(rows)
display(mp3_file_qc)

if (mp3_file_qc["missing_tracks"] > 0).any():
    raise FileNotFoundError(
        "12번 MP3 robustness 노트북에서 Test MP3 파일 생성 셀을 먼저 실행하세요."
    )

print("MP3 file QC PASS: True")

## 5. 13번 CNN 구조 및 checkpoint 로드

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2),
        )

    def forward(self, x):
        return self.block(x)


class LogMelCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            ConvBlock(1, 16),
            ConvBlock(16, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x.squeeze(1)


model = LogMelCNN().to(DEVICE)

checkpoint = torch.load(
    CNN_CHECKPOINT,
    map_location=DEVICE,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

total_params = sum(p.numel() for p in model.parameters())

with open(CNN_THRESHOLD_PATH, "r", encoding="utf-8") as f:
    thresholds = json.load(f)

SEGMENT_THRESHOLD = float(thresholds["segment_eer_threshold"])
TRACK_THRESHOLD = float(thresholds["track_eer_threshold"])

print("Best epoch       :", checkpoint["epoch"])
print("Total params     :", total_params)
print("Segment threshold:", SEGMENT_THRESHOLD)
print("Track threshold  :", TRACK_THRESHOLD)

assert total_params == 294321

## 6. MP3 segment → Log-Mel 함수

In [ ]:
def load_compressed_segment(
    audio_path: Path,
    start_sec: float,
):
    y, _ = librosa.load(
        audio_path,
        sr=SR,
        mono=True,
        offset=float(start_sec),
        duration=SEGMENT_SEC,
    )

    y = np.asarray(y, dtype=np.float32)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(
            y,
            (0, TARGET_SAMPLES - len(y)),
            mode="constant",
        )
    elif len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]

    return y


def waveform_to_logmel(y):
    mel = librosa.feature.melspectrogram(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=FMAX,
        power=2.0,
        center=True,
    )

    logmel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    # [-80, 0] -> [-1, 1]
    logmel = (logmel + 80.0) / 40.0 - 1.0

    logmel = np.asarray(logmel, dtype=np.float32)

    if logmel.shape[1] < N_FRAMES:
        logmel = np.pad(
            logmel,
            ((0, 0), (0, N_FRAMES - logmel.shape[1])),
            mode="constant",
            constant_values=-1.0,
        )
    elif logmel.shape[1] > N_FRAMES:
        logmel = logmel[:, :N_FRAMES]

    assert logmel.shape == (N_MELS, N_FRAMES)

    return logmel

## 7. 압축 조건별 Log-Mel cache 생성 — 128k / 64k

In [ ]:
def build_compressed_logmel_cache(condition):
    cache_path = (
        CNN_MP3_CACHE_DIR
        / f"logmel_test_{condition}_float16.npy"
    )
    done_path = (
        CNN_MP3_CACHE_DIR
        / f"logmel_test_{condition}_done.npy"
    )

    if cache_path.exists():
        cache = np.lib.format.open_memmap(
            cache_path,
            mode="r+",
            dtype=np.float16,
            shape=(len(test_segments), N_MELS, N_FRAMES),
        )
    else:
        cache = np.lib.format.open_memmap(
            cache_path,
            mode="w+",
            dtype=np.float16,
            shape=(len(test_segments), N_MELS, N_FRAMES),
        )

    if done_path.exists():
        done = np.load(done_path)
        if len(done) != len(test_segments):
            raise ValueError("done mask length mismatch")
    else:
        done = np.zeros(len(test_segments), dtype=bool)

    start_time = time.time()

    for i, row in test_segments.iterrows():
        if done[i]:
            continue

        audio_file = compressed_path(
            condition,
            row["track_sample_id"],
        )

        y = load_compressed_segment(
            audio_file,
            row["start_sec"],
        )

        logmel = waveform_to_logmel(y)

        cache[i] = logmel.astype(np.float16)
        done[i] = True

        if (i + 1) % 200 == 0 or (i + 1) == len(test_segments):
            cache.flush()
            np.save(done_path, done)

            elapsed = time.time() - start_time
            print(
                f"{condition}: {int(done.sum())}/{len(done)} "
                f"| elapsed {elapsed/60:.1f} min"
            )

    cache.flush()
    np.save(done_path, done)

    print(
        condition,
        "complete:",
        int(done.sum()),
        "/",
        len(done),
    )

    assert bool(done.all())

    return cache_path, done_path


compressed_cache_paths = {}

for condition in ["mp3_128", "mp3_64"]:
    print("\n" + "=" * 70)
    print("LOG-MEL CACHE:", condition)
    print("=" * 70)

    compressed_cache_paths[condition] = (
        build_compressed_logmel_cache(condition)
    )

print("Compressed Log-Mel Cache QC PASS: True")

## 8. 평가 Dataset / DataLoader

In [ ]:
class ConditionDataset(Dataset):
    def __init__(
        self,
        metadata,
        cache,
        cache_indices,
    ):
        self.metadata = metadata.reset_index(drop=True)
        self.cache = cache
        self.cache_indices = np.asarray(cache_indices, dtype=int)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, i):
        cache_idx = int(self.cache_indices[i])

        x = np.asarray(
            self.cache[cache_idx],
            dtype=np.float32,
        )

        x = torch.from_numpy(x).unsqueeze(0)

        y = torch.tensor(
            float(self.metadata.iloc[i]["label_id"]),
            dtype=torch.float32,
        )

        return x, y, i


def make_condition_loader(condition):
    if condition == "original":
        cache = original_cache
        indices = test_segments["global_index"].to_numpy()
    else:
        cache_path, _ = compressed_cache_paths[condition]
        cache = np.load(cache_path, mmap_mode="r")
        indices = np.arange(len(test_segments))

    ds = ConditionDataset(
        test_segments,
        cache,
        indices,
    )

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
    )

    return ds, loader

## 9. 예측 / 평가 함수

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    ys = []
    scores = []
    indices = []

    model.eval()

    for x, y, idx in loader:
        x = x.to(DEVICE)

        logits = model(x)
        prob = torch.sigmoid(logits)

        ys.append(y.numpy())
        scores.append(
            prob.detach().cpu().numpy()
        )
        indices.append(idx.numpy())

    return (
        np.concatenate(ys),
        np.concatenate(scores),
        np.concatenate(indices),
    )


def find_eer_threshold(y_true, scores):
    fpr, tpr, thresholds = roc_curve(
        y_true,
        scores,
        pos_label=1,
    )

    fnr = 1.0 - tpr
    valid = np.isfinite(thresholds)

    fpr = fpr[valid]
    fnr = fnr[valid]
    thresholds = thresholds[valid]

    idx = np.argmin(np.abs(fpr - fnr))

    return {
        "eer": float((fpr[idx] + fnr[idx]) / 2.0),
        "threshold": float(thresholds[idx]),
    }


def evaluate_scores(y_true, scores, threshold):
    y_true = np.asarray(y_true, dtype=int)
    scores = np.asarray(scores, dtype=float)

    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    ).ravel()

    eer_info = find_eer_threshold(
        y_true,
        scores,
    )

    return {
        "roc_auc": float(
            roc_auc_score(y_true, scores)
        ),
        "pr_auc": float(
            average_precision_score(y_true, scores)
        ),
        "eer": float(eer_info["eer"]),
        "balanced_accuracy": float(
            balanced_accuracy_score(y_true, y_pred)
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            )
        ),
        "real_fpr": float(fp / (fp + tn)),
        "fake_miss_rate": float(fn / (fn + tp)),
        "threshold_used": float(threshold),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def make_track_scores(metadata, scores):
    temp = metadata[
        [
            "track_sample_id",
            "original_audio",
            "label",
            "label_id",
            "genre",
            "generator",
            "split",
        ]
    ].copy()

    temp["score"] = scores

    return (
        temp
        .groupby("track_sample_id", as_index=False)
        .agg(
            original_audio=("original_audio", "first"),
            label=("label", "first"),
            label_id=("label_id", "first"),
            genre=("genre", "first"),
            generator=("generator", "first"),
            split=("split", "first"),
            segment_count=("score", "size"),
            score=("score", "mean"),
        )
    )

## 10. Original / MP3 128 / MP3 64 CNN 평가

In [ ]:
condition_results = []
track_outputs = {}
segment_outputs = {}

for condition in ["original", "mp3_128", "mp3_64"]:
    print("\n" + "=" * 70)
    print("EVALUATION:", condition)
    print("=" * 70)

    _, loader = make_condition_loader(condition)

    y, scores, local_indices = predict_loader(
        model,
        loader,
    )

    meta_ordered = (
        test_segments.iloc[local_indices]
        .reset_index(drop=True)
    )

    segment_metrics = evaluate_scores(
        y,
        scores,
        SEGMENT_THRESHOLD,
    )

    track_df = make_track_scores(
        meta_ordered,
        scores,
    )

    track_metrics = evaluate_scores(
        track_df["label_id"],
        track_df["score"],
        TRACK_THRESHOLD,
    )

    condition_results.append({
        "condition": condition,
        "level": "segment",
        **segment_metrics,
    })

    condition_results.append({
        "condition": condition,
        "level": "track",
        **track_metrics,
    })

    segment_outputs[condition] = (
        meta_ordered.copy(),
        scores.copy(),
    )
    track_outputs[condition] = (
        track_df.copy()
    )

cnn_mp3_metrics = pd.DataFrame(condition_results)

display(
    cnn_mp3_metrics[
        [
            "condition",
            "level",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
            "threshold_used",
        ]
    ].round(4)
)

## 11. Track-level 핵심 결과 및 성능 저하량

In [ ]:
cnn_track = (
    cnn_mp3_metrics[
        cnn_mp3_metrics["level"] == "track"
    ]
    .copy()
)

order = pd.CategoricalDtype(
    ["original", "mp3_128", "mp3_64"],
    ordered=True,
)

cnn_track["condition"] = (
    cnn_track["condition"].astype(order)
)

cnn_track = cnn_track.sort_values("condition")

display(
    cnn_track[
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].round(4)
)

orig = cnn_track[
    cnn_track["condition"] == "original"
].iloc[0]

drop_rows = []

for _, r in cnn_track.iterrows():
    drop_rows.append({
        "condition": r["condition"],
        "roc_auc_drop_vs_original":
            float(orig["roc_auc"] - r["roc_auc"]),
        "eer_change_vs_original":
            float(r["eer"] - orig["eer"]),
        "real_fpr_change_vs_original":
            float(r["real_fpr"] - orig["real_fpr"]),
        "fake_miss_change_vs_original":
            float(r["fake_miss_rate"] - orig["fake_miss_rate"]),
    })

compression_drop = pd.DataFrame(drop_rows)

print("Compression change vs Original")
display(compression_drop.round(4))

## 12. 13번 Original 결과와 일치하는지 QC

In [ ]:
expected_original = {
    "roc_auc": 0.9772,
    "eer": 0.1042,
    "balanced_accuracy": 0.9111,
    "macro_f1": 0.8535,
    "real_fpr": 0.1333,
    "fake_miss_rate": 0.0445,
}

original_row = cnn_track[
    cnn_track["condition"] == "original"
].iloc[0]

for k, expected in expected_original.items():
    print(
        k,
        "current =", round(float(original_row[k]), 4),
        "| expected ≈", expected,
    )

print(
    "Original result consistency:",
    all(
        abs(float(original_row[k]) - v) < 0.002
        for k, v in expected_original.items()
    )
)

## 13. Generator별 Track ROC-AUC

In [ ]:
def evaluate_generator_track(track_df):
    real_df = track_df[
        track_df["label"] == "REAL"
    ]

    fake_df = track_df[
        track_df["label"] == "FAKE"
    ]

    rows = []

    for generator in sorted(
        fake_df["generator"].dropna().unique()
    ):
        subgroup = pd.concat(
            [
                real_df,
                fake_df[
                    fake_df["generator"] == generator
                ],
            ],
            ignore_index=True,
        )

        rows.append({
            "generator": generator,
            "n_fake": int(
                (subgroup["label"] == "FAKE").sum()
            ),
            "roc_auc": float(
                roc_auc_score(
                    subgroup["label_id"],
                    subgroup["score"],
                )
            ),
        })

    return pd.DataFrame(rows)


generator_rows = []

for condition in ["original", "mp3_128", "mp3_64"]:
    temp = evaluate_generator_track(
        track_outputs[condition]
    )
    temp["condition"] = condition
    generator_rows.append(temp)

generator_metrics = pd.concat(
    generator_rows,
    ignore_index=True,
)

generator_auc_table = generator_metrics.pivot(
    index="generator",
    columns="condition",
    values="roc_auc",
)

display(generator_auc_table.round(4))

generator_auc_table["drop_64_vs_original"] = (
    generator_auc_table["original"]
    - generator_auc_table["mp3_64"]
)

print("\nLargest 64k AUC drops")
display(
    generator_auc_table
    .sort_values("drop_64_vs_original", ascending=False)
    .head(12)
    .round(4)
)

## 14. RBF-SVM vs CNN — MP3 robustness 비교

In [ ]:
if SVM_MP3_METRICS.exists():
    svm_all = pd.read_csv(SVM_MP3_METRICS)

    svm_track = svm_all[
        (svm_all["model"] == "RBF-SVM")
        & (svm_all["level"] == "track")
    ][
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    svm_track["model"] = "RBF-SVM"

    cnn_compare = cnn_track[
        [
            "condition",
            "roc_auc",
            "eer",
            "balanced_accuracy",
            "macro_f1",
            "real_fpr",
            "fake_miss_rate",
        ]
    ].copy()

    cnn_compare["model"] = "LogMelCNN"

    model_comparison = pd.concat(
        [svm_track, cnn_compare],
        ignore_index=True,
    )

    display(
        model_comparison[
            [
                "model",
                "condition",
                "roc_auc",
                "eer",
                "balanced_accuracy",
                "macro_f1",
                "real_fpr",
                "fake_miss_rate",
            ]
        ].round(4)
    )
else:
    print(
        "12번 SVM MP3 metrics 파일이 없어 "
        "CNN 결과만 저장합니다."
    )
    model_comparison = pd.DataFrame()

## 15. 결과 저장

In [ ]:
cnn_mp3_metrics.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_mp3_robustness_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

compression_drop.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_track_compression_change.csv",
    index=False,
    encoding="utf-8-sig",
)

generator_metrics.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_generator_compression_metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

generator_auc_table.to_csv(
    CNN_MP3_RESULT_DIR / "cnn_generator_compression_auc_table.csv",
    encoding="utf-8-sig",
)

if len(model_comparison):
    model_comparison.to_csv(
        CNN_MP3_RESULT_DIR / "svm_vs_cnn_mp3_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )

print("Saved to:", CNN_MP3_RESULT_DIR)

## 16. 최종 QC

In [ ]:
cache_128 = np.load(
    compressed_cache_paths["mp3_128"][0],
    mmap_mode="r",
)
cache_64 = np.load(
    compressed_cache_paths["mp3_64"][0],
    mmap_mode="r",
)

done_128 = np.load(
    compressed_cache_paths["mp3_128"][1]
)
done_64 = np.load(
    compressed_cache_paths["mp3_64"][1]
)

qc = pd.DataFrame({
    "check": [
        "test_segments",
        "test_tracks",
        "mp3_128_tracks",
        "mp3_64_tracks",
        "mp3_128_cache_rows",
        "mp3_64_cache_rows",
        "mp3_128_cache_complete",
        "mp3_64_cache_complete",
        "model_params",
        "result_rows",
        "track_conditions",
        "original_consistency",
    ],
    "value": [
        len(test_segments),
        len(test_tracks),
        int(mp3_file_qc.loc[
            mp3_file_qc["condition"] == "mp3_128",
            "existing_tracks"
        ].iloc[0]),
        int(mp3_file_qc.loc[
            mp3_file_qc["condition"] == "mp3_64",
            "existing_tracks"
        ].iloc[0]),
        cache_128.shape[0],
        cache_64.shape[0],
        bool(done_128.all()),
        bool(done_64.all()),
        total_params,
        len(cnn_mp3_metrics),
        cnn_track["condition"].nunique(),
        all(
            abs(float(original_row[k]) - v) < 0.002
            for k, v in expected_original.items()
        ),
    ],
})

display(qc)

core_qc_pass = (
    len(test_segments) == 1572
    and len(test_tracks) == 539
    and (mp3_file_qc["existing_tracks"] == 539).all()
    and cache_128.shape == (1572, 128, 1001)
    and cache_64.shape == (1572, 128, 1001)
    and bool(done_128.all())
    and bool(done_64.all())
    and total_params == 294321
    and len(cnn_mp3_metrics) == 6
    and cnn_track["condition"].nunique() == 3
    and all(
        abs(float(original_row[k]) - v) < 0.002
        for k, v in expected_original.items()
    )
)

print("===== FINAL RESULT =====")
print("CNN MP3 Robustness Core QC PASS:", core_qc_pass)

## 최종 확인 결과

- **1번**: `PyTorch`, `Device`
- **4번**: MP3 file QC
- **7번**: `Compressed Log-Mel Cache QC PASS`
- **10번**: Original / MP3 128 / MP3 64 전체 CNN 결과
- **11번**: Track-level 핵심 결과 + Original 대비 변화량
- **13번**: generator별 AUC 표
- **14번**: RBF-SVM vs CNN 비교표
- **16번**: `CNN MP3 Robustness Core QC PASS: True`

가장 중요한 비교는 Track-level:

```text
                 Original   MP3 128   MP3 64
RBF-SVM AUC       0.9644     0.9536   0.9062
Log-Mel CNN       0.9772     0.8954   0.8256
```

## 최신 실행 결과 요약 (2026-09-13)

Full-decode Log-Mel 기준 CNN Track-level 압축 강건성 결과다.

| 조건 | RBF-SVM ROC-AUC | Log-Mel CNN ROC-AUC | CNN EER |
|---|---:|---:|---:|
| Original | 0.9644 | **0.9772** | 0.1042 |
| MP3 128 kbps | **0.9536** | 0.8954 | 0.1830 |
| MP3 64 kbps | **0.9062** | 0.8256 | 0.2892 |

- CNN ROC-AUC는 Original 대비 128 kbps에서 0.0818, 64 kbps에서 0.1516 하락했다.
- CNN은 clean/in-domain에서는 가장 좋지만 codec shift에는 RBF-SVM보다 민감했다.
- Full-decode 결과를 `results/cnn_mp3_robustness/`에 저장했다.

**최종 상태: `CNN MP3 Robustness Core QC PASS = True`.**
